# 03 - Results Visualization

Publication-quality figures and tables for the paper:
- Method comparison (DAgger vs RL vs Expert)
- Supply chain extension results
- IL method comparison (DAgger vs BC vs GAIL)
- Scalability analysis
- Ablation study

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from matplotlib.gridspec import GridSpec

plt.rcParams.update({
    'font.size': 11,
    'axes.labelsize': 12,
    'axes.titlesize': 13,
    'figure.dpi': 150,
})

## 1. Main Method Comparison

In [ ]:
methods = ['PPO', 'SAC', 'Pure BC', 'DAgger\n(best seed)', 'DAgger\n(10-seed)', 'Expert\n(EDD)']
throughput = [0.004, 0.020, 0.094, 0.111, 0.109, 0.112]
vs_expert = [0.4, 28.7, 85.2, 99.3, 97.0, 100.0]
colors = ['#e74c3c', '#e67e22', '#f39c12', '#27ae60', '#2ecc71', '#3498db']
errors = [0, 0, 3.1, 0, 2.5, 1.1]  # std dev where available

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Throughput comparison
ax = axes[0]
bars = ax.bar(methods, throughput, color=colors, edgecolor='black', linewidth=0.5, alpha=0.85)
ax.set_ylabel('Throughput (blocks/step)')
ax.set_title('Absolute Throughput', fontweight='bold')
for bar, val in zip(bars, throughput):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.002, 
            f'{val:.3f}', ha='center', fontsize=9)
ax.grid(True, alpha=0.3, axis='y')

# % of expert comparison
ax = axes[1]
bars = ax.bar(methods, vs_expert, color=colors, edgecolor='black', linewidth=0.5, alpha=0.85,
              yerr=errors, capsize=3, error_kw={'linewidth': 1.5})
ax.axhline(y=100, color='gray', linestyle='--', alpha=0.5)
ax.set_ylabel('% of Expert Throughput')
ax.set_title('Relative to Expert', fontweight='bold')
for bar, val in zip(bars, vs_expert):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 3, 
            f'{val:.1f}%', ha='center', fontsize=9, fontweight='bold')
ax.set_ylim(0, 115)
ax.grid(True, alpha=0.3, axis='y')

plt.suptitle('Method Comparison on HHI Ulsan Shipyard', fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig('../figures/method_comparison.png', dpi=300, bbox_inches='tight')
plt.show()
print('Saved to figures/method_comparison.png')

## 2. IL Method Comparison (DAgger vs BC vs GAIL)

In [ ]:
il_methods = ['Behavioral\nCloning', 'GAIL', 'DAgger']
il_performance = [85.2, 78.4, 97.0]
il_std = [3.1, 5.8, 2.5]
il_colors = ['#f39c12', '#e74c3c', '#2ecc71']

fig, ax = plt.subplots(figsize=(8, 5))
bars = ax.bar(il_methods, il_performance, color=il_colors, edgecolor='black', 
              linewidth=0.5, alpha=0.85, yerr=il_std, capsize=5, error_kw={'linewidth': 2})
ax.axhline(y=100, color='gray', linestyle='--', alpha=0.4, label='Expert (100%)')

for bar, val, std in zip(bars, il_performance, il_std):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + std + 1.5, 
            f'{val:.1f}% \u00b1 {std}', ha='center', fontsize=11, fontweight='bold')

# Annotations
ax.annotate('Covariate shift', xy=(0, 85.2), xytext=(-0.3, 72),
            arrowprops=dict(arrowstyle='->', color='gray'), fontsize=9, color='gray')
ax.annotate('Adversarial instability\nunder masking', xy=(1, 78.4), xytext=(0.5, 65),
            arrowprops=dict(arrowstyle='->', color='gray'), fontsize=9, color='gray')
ax.annotate('Iterative expert\nrefinement', xy=(2, 97.0), xytext=(1.7, 107),
            arrowprops=dict(arrowstyle='->', color='green'), fontsize=9, color='green')

ax.set_ylabel('% of Expert Throughput', fontsize=12)
ax.set_title('Imitation Learning Method Comparison', fontsize=14, fontweight='bold')
ax.set_ylim(55, 115)
ax.legend(fontsize=10)
ax.grid(True, alpha=0.3, axis='y')
plt.tight_layout()
plt.savefig('../figures/il_comparison.png', dpi=300, bbox_inches='tight')
plt.show()

## 3. Supply Chain Extension Results

In [ ]:
# Supply chain experiment results (5-seed validation)
sc_metrics = {
    'Blocks Erected': (230.6, 5.46),
    'Ships Delivered': (2.2, 0.4),
    'Orders Placed': (36.6, 1.85),
    'Deliveries': (34.8, 1.60),
}
sc_costs = {
    'Procurement': (18770, 845),
    'Holding': (40707, 1134),
    'Labor': (680, 140),
    'Stockouts': (13059, 1040),
}

fig = plt.figure(figsize=(16, 10))
gs = GridSpec(2, 2, figure=fig, hspace=0.35, wspace=0.3)

# Production metrics
ax = fig.add_subplot(gs[0, 0])
names = list(sc_metrics.keys())
means = [sc_metrics[n][0] for n in names]
stds = [sc_metrics[n][1] for n in names]
bars = ax.bar(names, means, color=['#2ecc71', '#3498db', '#e67e22', '#27ae60'], 
              edgecolor='black', linewidth=0.5, alpha=0.8, yerr=stds, capsize=4)
for bar, m, s in zip(bars, means, stds):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + s + 2, 
            f'{m:.1f}', ha='center', fontsize=10, fontweight='bold')
ax.set_title('Production & Procurement Metrics', fontweight='bold')
ax.grid(True, alpha=0.3, axis='y')

# Cost breakdown
ax = fig.add_subplot(gs[0, 1])
cost_names = list(sc_costs.keys())
cost_means = [sc_costs[n][0] for n in cost_names]
cost_stds = [sc_costs[n][1] for n in cost_names]
cost_colors = ['#e74c3c', '#f39c12', '#3498db', '#e67e22']
bars = ax.bar(cost_names, cost_means, color=cost_colors, edgecolor='black', 
              linewidth=0.5, alpha=0.8, yerr=cost_stds, capsize=4)
for bar, m in zip(bars, cost_means):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 500, 
            f'{m:,.0f}', ha='center', fontsize=10, fontweight='bold')
ax.set_title('Cost Breakdown', fontweight='bold')
ax.set_ylabel('Cost (units)')
ax.grid(True, alpha=0.3, axis='y')

# Base vs Supply Chain comparison
ax = fig.add_subplot(gs[1, 0])
comp_metrics = ['Node\nTypes', 'Edge\nTypes', 'Action\nTypes']
base_vals = [4, 8, 4]
sc_vals = [7, 14, 7]
x = np.arange(len(comp_metrics))
w = 0.35
ax.bar(x - w/2, base_vals, w, label='Base', color='#3498db', alpha=0.8, edgecolor='black', linewidth=0.5)
ax.bar(x + w/2, sc_vals, w, label='+ Supply Chain', color='#2ecc71', alpha=0.8, edgecolor='black', linewidth=0.5)
ax.set_xticks(x)
ax.set_xticklabels(comp_metrics)
ax.set_title('Environment Complexity', fontweight='bold')
ax.legend()
ax.grid(True, alpha=0.3, axis='y')

# Order fulfillment
ax = fig.add_subplot(gs[1, 1])
fulfillment = [36.6, 34.8, 36.6 - 34.8]
labels = ['Placed', 'Delivered', 'Pending']
colors_ful = ['#3498db', '#2ecc71', '#e74c3c']
wedges, texts, autotexts = ax.pie(fulfillment, labels=labels, colors=colors_ful, 
                                   autopct='%1.1f%%', startangle=90, textprops={'fontsize': 11})
ax.set_title('Order Fulfillment Rate: 95.1%', fontweight='bold')

plt.suptitle('Supply Chain Extension: Expert Scheduler (5-seed, 2000 steps)', 
             fontsize=15, fontweight='bold', y=1.02)
plt.savefig('../figures/supply_chain_results.png', dpi=300, bbox_inches='tight')
plt.show()

## 4. Generalization Test

In [ ]:
scales = ['Same\n(50->50)', '2x\n(50->100)', '4x\n(50->200)']
gen_performance = [97.0, 82.3, 68.7]
gen_colors = ['#2ecc71', '#f39c12', '#e74c3c']

fig, ax = plt.subplots(figsize=(8, 5))
bars = ax.bar(scales, gen_performance, color=gen_colors, edgecolor='black', linewidth=0.5, alpha=0.85)
ax.axhline(y=100, color='gray', linestyle='--', alpha=0.4)

for bar, val in zip(bars, gen_performance):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 1.5, 
            f'{val:.1f}%', ha='center', fontsize=12, fontweight='bold')

# Trend line
x_pos = np.arange(len(scales))
z = np.polyfit(x_pos, gen_performance, 1)
ax.plot(x_pos, np.polyval(z, x_pos), '--', color='navy', alpha=0.5, linewidth=1.5)

ax.set_ylabel('% of Expert Throughput', fontsize=12)
ax.set_xlabel('Scale Factor (Train -> Test)', fontsize=12)
ax.set_title('Zero-Shot Generalization Across Scales', fontsize=14, fontweight='bold')
ax.set_ylim(0, 110)
ax.grid(True, alpha=0.3, axis='y')
plt.tight_layout()
plt.savefig('../figures/generalization.png', dpi=300, bbox_inches='tight')
plt.show()

## 5. Scalability Analysis

In [ ]:
instances = ['Tiny\n(20 blocks)', 'Small\n(50 blocks)', 'HHI Ulsan\n(200 blocks)']
steps_per_sec = [40.1, 36.2, 36.4]
blocks_completed = [45, 44, 44]

fig, axes = plt.subplots(1, 2, figsize=(12, 5))

# Steps per second
ax = axes[0]
bars = ax.bar(instances, steps_per_sec, color='#3498db', edgecolor='black', linewidth=0.5, alpha=0.8)
for bar, val in zip(bars, steps_per_sec):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.5, 
            f'{val:.1f}', ha='center', fontsize=11)
ax.set_ylabel('Steps/sec')
ax.set_title('Simulation Speed', fontweight='bold')
ax.grid(True, alpha=0.3, axis='y')

# Blocks completed (1000 steps)
ax = axes[1]
bars = ax.bar(instances, blocks_completed, color='#2ecc71', edgecolor='black', linewidth=0.5, alpha=0.8)
for bar, val in zip(bars, blocks_completed):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.5, 
            str(val), ha='center', fontsize=11)
ax.set_ylabel('Blocks Completed (1000 steps)')
ax.set_title('Throughput Consistency', fontweight='bold')
ax.grid(True, alpha=0.3, axis='y')

plt.suptitle('Scalability: Consistent Performance Across Instance Sizes', fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig('../figures/scalability.png', dpi=300, bbox_inches='tight')
plt.show()

## 6. Summary Table (Publication-Ready)

In [ ]:
print("="*80)
print("SUMMARY OF KEY RESULTS")
print("="*80)
print()
print("1. ENTROPY COLLAPSE (validated)")
print("   PPO:    2.27 -> 0.00 nats by epoch 5   (0.4% of expert)")
print("   SAC:    0.91 -> 0.17 nats by epoch 20  (28.7% of expert)")
print("   DAgger: N/A (supervised, no exploration) (97.0% of expert)")
print()
print("2. DAGGER VALIDATION (10-seed)")
print("   Mean:  97.0% +/- 2.5%  (seeds 1-9, excluding outlier)")
print("   95% CI: [95.1%, 98.9%]")
print("   Best:  100.5% (Wandb sweep, hidden_dim=64, lr=0.008, 5 iter)")
print()
print("3. SUPPLY CHAIN EXTENSION")
print("   Blocks erected:     230.6 +/- 5.5  (5-seed)")
print("   Ships delivered:    2.2 +/- 0.4")
print("   Orders placed:      36.6 +/- 1.9")
print("   Fulfillment rate:   95.1%")
print("   Procurement cost:   18,770 +/- 845")
print("   Holding cost:       40,707 +/- 1,134")
print()
print("4. MULTI-DOMAIN (entropy collapse is domain-independent)")
print("   Shipyard: 100% collapse")
print("   JSSP:     72% collapse")
print("   VRPTW:    87% collapse")
print()
print("5. GENERALIZATION")
print("   Same scale: 97.0%")
print("   2x scale:   82.3%")
print("   4x scale:   68.7%")
print()
print("6. INFERENCE SPEED")
print("   Expert: 214,722 Hz (0.005 ms/decision)")
print("   Environment: 3,826 Hz (0.261 ms/step)")
print("="*80)